In [33]:
!pip install pandas

In [2]:
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/PeiyaoPearl/CASA0006-Data-Science-for-Spatial-Systems-/refs/heads/main/Assessments/data/LTDS_Data_2014_2020.csv"

df = pd.read_csv(url, encoding='latin-1')

df.head()

,Unique trip ID,Trip weight,Year,Mode,Journey purpose,Distance (km),Gender,Age,Car access,Working status,Household structure,Disability,Trip start time period,Origin borough,Origin postcode sector,Destination borough,Destination postcode sector,Ethnicity,Household income
0,190010210101,217.798842,2019/20,01 Walk (roller blades/ scooters),04 Shopping and personal business,1.070,Female,32,00 No car,Not working,Single pensioner,Not disabled,AM peak (0700-0959),15 Barking & Dagenham,RM9 4,15 Barking & Dagenham,IG11 9,"02 Mixed, Other and Arab","03 ?10,000 - ?14,999"
1,190010210102,217.798842,2019/20,01 Walk (roller blades/ scooters),04 Shopping and personal business,1.070,Female,32,00 No car,Not working,Single pensioner,Not disabled,Interpeak (1000-1559),15 Barking & Dagenham,IG11 9,15 Barking & Dagenham,RM9 4,"02 Mixed, Other and Arab","03 ?10,000 - ?14,999"
2,190010510301,276.187543,2019/20,17 Underground,01 Usual workplace,15.193,Male,35,02 Two or more car,Full-time worker,Couple without children,Not disabled,AM peak (0700-0959),15 Barking & Dagenham,RM9 4,06 Islington,N1 8,03 Asian,"09 ?75,000 - ?99,999"
3,190010510302,276.187543,2019/20,17 Underground,01 Usual workplace,15.193,Male,35,02 Two or more car,Full-time worker,Couple without children,Not disabled,Evening (1900-2159),06 Islington,N1 8,15 Barking & Dagenham,RM9 4,03 Asian,"09 ?75,000 - ?99,999"
4,190010710101,694.504624,2019/20,17 Underground,03 Education,15.914,Male,55,01 One car,Part-time worker,Lone parent,Not disabled,AM peak (0700-0959),15 Barking & Dagenham,RM9 4,11 Southwark,SE1 0,03 Asian,"02 ?5,000 - ?9,999"


In [3]:
df.columns = df.columns.str.lower().str.strip().str.replace(' ', '_')
#remove code
def clean_label_v2(col_name):
    return (df[col_name].astype(str)
            .str.replace('?', '£', regex=False) 
            .str.split(' ', n=1).str[1]
            .str.lower()
            .str.strip())

df['car_access'] = clean_label_v2('car_access')
df['household_income'] = clean_label_v2('household_income')
df['journey_purpose'] = clean_label_v2('journey_purpose')
df['ethnicity'] = clean_label_v2('ethnicity')
df['origin_borough'] = clean_label_v2('origin_borough')

df['journey_purpose'] = df['journey_purpose'].replace('missing', 'other')
df['household_income'] = df['household_income'].astype(str).str.replace('£', '').str.replace(',', '')

print(df[['household_income', 'car_access', 'journey_purpose']].head())

# mode classification
df['mode_code'] = df['mode'].str.extract('(\d+)').astype(float)

def map_mode_group(code):
    if code == 1:
        return 'walk'
    elif code == 2:
        return 'cycle'
    elif code in [3, 4, 5, 6, 7, 8, 9, 10, 11, 12]: 
        return 'car'
    elif code in [13, 14, 15, 16, 17, 18, 19, 20, 24, 25]: 
        return 'pt'

df['mode'] = df['mode_code'].apply(map_mode_group)

#year
df['year'] = df['year'].astype(str).str[:4].astype(int)

#lower
df['gender'] = df['gender'].astype(str).str.lower().str.strip()
df['working_status'] = df['working_status'].astype(str).str.lower().str.strip()
df['household_structure'] = df['household_structure'].astype(str).str.lower().str.strip()
df['disability'] = df['disability'].astype(str).str.lower().str.strip()
df['trip_start_time_period'] = df['trip_start_time_period'].astype(str).str.lower().str.strip()
df['origin_borough'] = df['origin_borough'].astype(str).str.lower().str.strip()

#Convert numeric values

df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['distance_(km)'] = pd.to_numeric(df['distance_(km)'], errors='coerce')

df.head()

# --- filter na ---
initial_count = len(df)

critical_cols = ['year', 'mode', 'journey_purpose', 'unique_trip_id', 'origin_borough', 'household_income','distance_(km)','gender', 'age', 'car_access', 'working_status', 'household_structure', 'disability', 'trip_start_time_period', 'ethnicity', 'household_income']
df = df.dropna(subset=critical_cols)

final_model_cols = [
    'unique_trip_id','year', 'mode', 'journey_purpose', 'distance_(km)','trip_start_time_period', 'origin_borough', 'household_income','gender', 'age', 'car_access', 'working_status', 'household_structure', 'disability', 'ethnicity', 'household_income'
]



df_final = df[final_model_cols].copy().reset_index(drop=True)

# --- report---
final_count = len(df_final)
print(f"cleaning done")
print(f"initial count: {initial_count}")
print(f"final count: {final_count}")
print(f"delete: {initial_count - final_count} row")
df_final.head()


<>:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
/var/folders/2f/9fqrm6p16sxbkj608dl5djbm0000gn/T/ipykernel_36482/2790845309.py:22: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  df['mode_code'] = df['mode'].str.extract('(\d+)').astype(float)


  household_income       car_access                 journey_purpose
0    10000 - 14999           no car  shopping and personal business
1    10000 - 14999           no car  shopping and personal business
2    75000 - 99999  two or more car                 usual workplace
3    75000 - 99999  two or more car                 usual workplace
4      5000 - 9999          one car                       education
cleaning done
initial count: 233622
final count: 230261
delete: 3361 row


,unique_trip_id,year,mode,journey_purpose,distance_(km),trip_start_time_period,origin_borough,household_income,gender,age,car_access,working_status,household_structure,disability,ethnicity,household_income
0,190010210101,2019,walk,shopping and personal business,1.070,am peak (0700-0959),barking & dagenham,10000 - 14999,female,32,no car,not working,single pensioner,not disabled,"mixed, other and arab",10000 - 14999
1,190010210102,2019,walk,shopping and personal business,1.070,interpeak (1000-1559),barking & dagenham,10000 - 14999,female,32,no car,not working,single pensioner,not disabled,"mixed, other and arab",10000 - 14999
2,190010510301,2019,pt,usual workplace,15.193,am peak (0700-0959),barking & dagenham,75000 - 99999,male,35,two or more car,full-time worker,couple without children,not disabled,asian,75000 - 99999
3,190010510302,2019,pt,usual workplace,15.193,evening (1900-2159),islington,75000 - 99999,male,35,two or more car,full-time worker,couple without children,not disabled,asian,75000 - 99999
4,190010710101,2019,pt,education,15.914,am peak (0700-0959),barking & dagenham,5000 - 9999,male,55,one car,part-time worker,lone parent,not disabled,asian,5000 - 9999


In [4]:
#load ptal csv
ptal_url = "https://raw.githubusercontent.com/PeiyaoPearl/CASA0006-Data-Science-for-Spatial-Systems-/refs/heads/main/Assessments/data/MSOA_aggregated_PTAL_stats_2023.csv"
ptal_msoa = pd.read_csv(ptal_url)
pop_url ="https://raw.githubusercontent.com/PeiyaoPearl/CASA0006-Data-Science-for-Spatial-Systems-/refs/heads/main/Assessments/data/msoa-population-.csv"
df_pop = pd.read_csv(pop_url)
print("done ！")
print(df.head())

# --- 2. 从 MSOA21NM 提取 Borough 名称 ---
# 例如 "Barking and Dagenham 001" -> 提取除最后三个数字和空格以外的部分
ptal_msoa['borough_from_msoa'] = ptal_msoa['MSOA21NM'].str.replace(r'\s\d+$', '', regex=True).str.lower().str.strip()

# 特殊处理：将 PTAL 表中的 "and" 替换为 "&" 以匹配你的 LTDS 表
ptal_msoa['borough_from_msoa'] = ptal_msoa['borough_from_msoa'].str.replace(' and ', ' & ')

# --- 3. 聚合到 Borough 级别 ---
# 我们取平均可达性指标 mean_AI 的均值，作为该 Borough 的交通供给水平
borough_ptal = ptal_msoa.groupby('borough_from_msoa')['mean_AI'].mean().reset_index()
borough_ptal.columns = ['origin_borough', 'borough_mean_ptal']

# --- 4. 与你的主表 df_final 合并 ---
# 确保你的 df_final['origin_borough'] 已经是小写且去掉了编号（使用之前写的 clean_label 函数）
df_final = pd.merge(df_final, borough_ptal, on='origin_borough', how='left')

# --- 5. 检查匹配情况 ---
missing_ptal = df_final[df_final['borough_mean_ptal'].isnull()]['origin_borough'].unique()
if len(missing_ptal) > 0:
    print(f"⚠️ 以下 Borough 未能匹配到 PTAL 数据，请检查名称映射：{missing_ptal}")
else:
    print("✅ 所有 Borough 的 PTAL 数据匹配成功！")

df_final.head()


done ！
   unique_trip_id  trip_weight  year  mode                 journey_purpose  \
0    190010210101   217.798842  2019  walk  shopping and personal business   
1    190010210102   217.798842  2019  walk  shopping and personal business   
2    190010510301   276.187543  2019    pt                 usual workplace   
3    190010510302   276.187543  2019    pt                 usual workplace   
4    190010710101   694.504624  2019    pt                       education   

   distance_(km)  gender  age       car_access    working_status  \
0          1.070  female   32           no car       not working   
1          1.070  female   32           no car       not working   
2         15.193    male   35  two or more car  full-time worker   
3         15.193    male   35  two or more car  full-time worker   
4         15.914    male   55          one car  part-time worker   

       household_structure    disability trip_start_time_period  \
0         single pensioner  not disabled    am p

,unique_trip_id,year,mode,journey_purpose,distance_(km),trip_start_time_period,origin_borough,household_income,gender,age,car_access,working_status,household_structure,disability,ethnicity,household_income,borough_mean_ptal
0,190010210101,2019,walk,shopping and personal business,1.070,am peak (0700-0959),barking & dagenham,10000 - 14999,female,32,no car,not working,single pensioner,not disabled,"mixed, other and arab",10000 - 14999,8.417092
1,190010210102,2019,walk,shopping and personal business,1.070,interpeak (1000-1559),barking & dagenham,10000 - 14999,female,32,no car,not working,single pensioner,not disabled,"mixed, other and arab",10000 - 14999,8.417092
2,190010510301,2019,pt,usual workplace,15.193,am peak (0700-0959),barking & dagenham,75000 - 99999,male,35,two or more car,full-time worker,couple without children,not disabled,asian,75000 - 99999,8.417092
3,190010510302,2019,pt,usual workplace,15.193,evening (1900-2159),islington,75000 - 99999,male,35,two or more car,full-time worker,couple without children,not disabled,asian,75000 - 99999,25.404079
4,190010710101,2019,pt,education,15.914,am peak (0700-0959),barking & dagenham,5000 - 9999,male,55,one car,part-time worker,lone parent,not disabled,asian,5000 - 9999,8.417092


In [5]:
# 1. 定义需要保留的“非法”标签列表（即那些不是真正区名的字符串）
invalid_labels = [
    'outside london area (m25)', 
    'london area, outside greater london', 
    'london area, outside g london',
    'missing',
    'ouside london area (m25)',
    'unknown'
]

# 2. 只保留不在这个列表里的行
# 这样剩下的就全是像 'barking & dagenham', 'westminster' 这种真正的行政区了
df_final = df_final[~df_final['origin_borough'].isin(invalid_labels)]

# 3. 再次检查，现在应该只剩下 32 个区 + City of London
print(df_final['origin_borough'].unique())

<StringArray>
[  'barking & dagenham',            'islington',            'southwark',
       'city of london',          'westminster',            'redbridge',
             'havering',              'hackney',              'croydon',
               'camden',        'tower hamlets',               'newham',
           'hillingdon',       'waltham forest',            'greenwich',
              'bromley',              'lambeth',               'harrow',
               'ealing',              'enfield',             'haringey',
           'wandsworth',               'barnet',                'brent',
             'hounslow', 'kensington & chelsea', 'richmond upon thames',
 'hammersmith & fulham',               'bexley',             'lewisham',
               'sutton',               'merton', 'kingston upon thames']
Length: 33, dtype: str


In [6]:
import pandas as pd

# 1. 加载数据
pop_url = "https://raw.githubusercontent.com/PeiyaoPearl/CASA0006-Data-Science-for-Spatial-Systems-/refs/heads/main/Assessments/data/msoa-population-.csv"
df_pop = pd.read_csv(pop_url)
print("done ！")

# --- 2. 提取 Borough 名称 ---
df_pop['borough_from_pop'] = df_pop['MSOA name'].str.replace(r'\s\d+$', '', regex=True).str.lower().str.strip()
df_pop['borough_from_pop'] = df_pop['borough_from_pop'].str.replace(' and ', ' & ')

# --- 3. 数据类型转换 (处理逗号并转为浮点数) ---
# 处理人口密度
df_pop['People per Sq Km'] = (
    df_pop['People per Sq Km'].astype(str)
    .str.replace(',', '').str.strip().astype(float)
)

# 处理面积 (同样预防有逗号的情况)
df_pop['Area Sq Km'] = (
    df_pop['Area Sq Km'].astype(str)
    .str.replace(',', '').str.strip().astype(float)
)

# --- 4. 聚合计算：密度取平均，面积取总和 ---
borough_stats = df_pop.groupby('borough_from_pop').agg({
    'People per Sq Km': 'mean',
    'Area Sq Km': 'sum'
}).reset_index()

# 重命名列名
borough_stats.columns = ['borough_from_pop', 'borough_pop_density', 'borough_total_area_sqkm']

# --- 5. 合并到主表 df_final ---
# 检查并删除旧列，防止重复运行报错 (增加了面积列的删除)
cols_to_drop = ['borough_from_pop', 'borough_pop_density', 'borough_total_area_sqkm']
df_final = df_final.drop(columns=[c for c in cols_to_drop if c in df_final.columns], errors='ignore')

df_final = pd.merge(
    df_final,
    borough_stats,
    left_on='origin_borough',
    right_on='borough_from_pop',
    how='left'
)

# --- 6. 检查结果 ---
print("✅ 人口密度与面积数据聚合完成！")
# 打印前几行看看面积是否合理（伦敦各区面积通常在 10-150 sqkm 之间）
print(df_final[['origin_borough', 'borough_pop_density', 'borough_total_area_sqkm']].drop_duplicates().head(5))

# 检查匹配失败
missing = df_final[df_final['borough_pop_density'].isnull()]['origin_borough'].unique()
if len(missing) > 0:
    print(f"⚠️ 警告：以下 Borough 未匹配成功: {missing}")
else:
    print("✅ 所有 Borough 数据匹配成功！")

done ！
✅ 人口密度与面积数据聚合完成！
        origin_borough  borough_pop_density  borough_total_area_sqkm
0   barking & dagenham          6530.500000                    36.09
3            islington         15129.695652                    14.87
5            southwark         12267.575758                    28.86
7       city of london          2786.000000                     2.90
18         westminster         15456.375000                    21.44
✅ 所有 Borough 数据匹配成功！


In [7]:
df_final.head()

,unique_trip_id,year,mode,journey_purpose,distance_(km),trip_start_time_period,origin_borough,household_income,gender,age,car_access,working_status,household_structure,disability,ethnicity,household_income,borough_mean_ptal,borough_from_pop,borough_pop_density,borough_total_area_sqkm
0,190010210101,2019,walk,shopping and personal business,1.070,am peak (0700-0959),barking & dagenham,10000 - 14999,female,32,no car,not working,single pensioner,not disabled,"mixed, other and arab",10000 - 14999,8.417092,barking & dagenham,6530.500000,36.09
1,190010210102,2019,walk,shopping and personal business,1.070,interpeak (1000-1559),barking & dagenham,10000 - 14999,female,32,no car,not working,single pensioner,not disabled,"mixed, other and arab",10000 - 14999,8.417092,barking & dagenham,6530.500000,36.09
2,190010510301,2019,pt,usual workplace,15.193,am peak (0700-0959),barking & dagenham,75000 - 99999,male,35,two or more car,full-time worker,couple without children,not disabled,asian,75000 - 99999,8.417092,barking & dagenham,6530.500000,36.09
3,190010510302,2019,pt,usual workplace,15.193,evening (1900-2159),islington,75000 - 99999,male,35,two or more car,full-time worker,couple without children,not disabled,asian,75000 - 99999,25.404079,islington,15129.695652,14.87
4,190010710101,2019,pt,education,15.914,am peak (0700-0959),barking & dagenham,5000 - 9999,male,55,one car,part-time worker,lone parent,not disabled,asian,5000 - 9999,8.417092,barking & dagenham,6530.500000,36.09


In [8]:
import pandas as pd

# 1. 从 GitHub 加载新下载的表格 (请替换为你实际的 raw 链接)
# 如果文件是 CSV 格式
road_csv_url = "https://raw.githubusercontent.com/PeiyaoPearl/CASA0006-Data-Science-for-Spatial-Systems-/refs/heads/main/Assessments/data/road_length.csv"
df_road_new = pd.read_csv(road_csv_url)

# 2. 字段预处理：将 Local Authority 换成小写并与主表 origin_borough 格式统一
# 我们会处理：转小写、去除两端空格、将 " and " 替换为 " & "
df_road_new['origin_borough'] = (df_road_new['Local Authority']
                                 .astype(str)
                                 .str.lower()
                                 .str.strip()
                                 .str.replace(' and ', ' & ', regex=False))

print(df_road_new.columns)

# 3. 提取核心列：行政区和总道路长度
# 注意：请根据你下载的表格实际表头确认是 'Total road length' 还是 'Total'
# 如果表头包含单位，请确保名称完全一致
road_col_name = 'Total road length ' 
# 仅保留需要的两列
df_road_subset = df_road_new[['origin_borough', road_col_name]].copy()

# 4. 与主表 df_final 匹配合并
# 先删除主表可能存在的旧列，防止重复
df_final = df_final.drop(columns=[road_col_name, 'road_density'], errors='ignore')

df_final = pd.merge(
    df_final,
    df_road_subset,
    on='origin_borough',
    how='left'
)

# 5. 计算路网密度 (Road Density)
# 公式：总长度 / 之前算好的面积
df_final['road_density'] = df_final[road_col_name] / df_final['borough_total_area_sqkm']

# 6. 检查匹配情况
print("✅ 道路长度数据匹配完成！")
missing_count = df_final[road_col_name].isnull().sum()
if missing_count > 0:
    print(f"⚠️ 注意：有 {missing_count} 行数据未能匹配到道路长度，请检查行政区名称是否一致。")
else:
    print("✨ 所有数据完美匹配！")

# 提取唯一的行政区及其路网密度，并按密度从高到低排序
borough_density_list = (df_final[['origin_borough', 'road_density']]
                        .drop_duplicates()
                        .sort_values(by='road_density', ascending=False)
                        .reset_index(drop=True))

# 打印完整列表（确保不被省略）
print(borough_density_list.to_string())

Index(['ONS Area \nCode', 'Region', 'Local Authority', 'Total road length ',
       'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8',
       'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12',
       'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16',
       'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20',
       'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24',
       'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28',
       'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32',
       'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36',
       'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40',
       'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44',
       'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48',
       'Unnamed: 49', 'Unnamed: 50', 'Unnamed: 51', 'Unnamed: 52',
       'Unnamed: 53', 'Unnamed: 54', 'Unnamed: 55', 'Unnamed: 56',
       'Unnamed: 57', 'Unnamed: 58', 'Unnam

In [9]:
df_final.head()

,unique_trip_id,year,mode,journey_purpose,distance_(km),trip_start_time_period,origin_borough,household_income,gender,age,...,household_structure,disability,ethnicity,household_income,borough_mean_ptal,borough_from_pop,borough_pop_density,borough_total_area_sqkm,Total road length,road_density
0,190010210101,2019,walk,shopping and personal business,1.070,am peak (0700-0959),barking & dagenham,10000 - 14999,female,32,...,single pensioner,not disabled,"mixed, other and arab",10000 - 14999,8.417092,barking & dagenham,6530.500000,36.09,346,9.587143
1,190010210102,2019,walk,shopping and personal business,1.070,interpeak (1000-1559),barking & dagenham,10000 - 14999,female,32,...,single pensioner,not disabled,"mixed, other and arab",10000 - 14999,8.417092,barking & dagenham,6530.500000,36.09,346,9.587143
2,190010510301,2019,pt,usual workplace,15.193,am peak (0700-0959),barking & dagenham,75000 - 99999,male,35,...,couple without children,not disabled,asian,75000 - 99999,8.417092,barking & dagenham,6530.500000,36.09,346,9.587143
3,190010510302,2019,pt,usual workplace,15.193,evening (1900-2159),islington,75000 - 99999,male,35,...,couple without children,not disabled,asian,75000 - 99999,25.404079,islington,15129.695652,14.87,237,15.938130
4,190010710101,2019,pt,education,15.914,am peak (0700-0959),barking & dagenham,5000 - 9999,male,55,...,lone parent,not disabled,asian,5000 - 9999,8.417092,barking & dagenham,6530.500000,36.09,346,9.587143


In [10]:

#export new csv
final_cols = [
    # --- target ---
    'unique_trip_id', 'year','mode',  

    # --- 行程特征 ---
    'distance_(km)', 'journey_purpose', 'trip_start_time_period', 'origin_borough', 

    # --- 个体/家庭特征 (Categorical/Numerical) ---
    'gender', 'age', 'car_access', 'working_status', 
    'household_income', 'disability', 'ethnicity', 'household_structure',
    
    # --- 空间环境特征 (刚才辛苦算出来的) ---
    'borough_mean_ptal', 
    'borough_pop_density', 
    'road_density',
]

df_clean = df_final[final_cols].copy()

# 检查各列缺失值
print("各列缺失值统计：")
print(df_clean.isnull().sum())

# 如果有少量缺失值，建议直接删掉（或者根据你的逻辑填充）
df_clean = df_clean.dropna()
print(f"\n清理后剩余行数: {len(df_clean)}")


# 导出到本地（如果你在本地运行）
df_clean.to_csv('ltds_clean.csv', index=False)

print("✅ 建模数据集已成功导出为 'ltds_clean.csv'")

各列缺失值统计：
unique_trip_id            0
year                      0
mode                      0
distance_(km)             0
journey_purpose           0
trip_start_time_period    0
origin_borough            0
gender                    0
age                       0
car_access                0
working_status            0
household_income          0
household_income          0
disability                0
ethnicity                 0
household_structure       0
borough_mean_ptal         0
borough_pop_density       0
road_density              0
dtype: int64

清理后剩余行数: 220183
✅ 建模数据集已成功导出为 'ltds_clean.csv'
